In [3]:
import cv2
import numpy as np
from ultralytics import YOLO
import mediapipe as mp

In [19]:
print(np.__version__)

2.2.6


In [24]:
# ===================== CONFIG =====================
IMAGE_PATH = r"assets/test3.jpg"       # مسیر عکس
CONF_THRESH = 0.6              # حداقل confidence برای YOLO
MAX_DIM = 800                  # حداکثر طول یا عرض تصویر بعد از resize

# Sitting thresholds
KNEE_ANGLE_SIT = 120
HIP_KNEE_RATIO_SIT = 1.2

# ===================== INIT MODELS =====================
yolo = YOLO(r"model/yolov8n.pt")   # فقط برای تشخیص person

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=True,
    model_complexity=1,
    enable_segmentation=False,
    min_detection_confidence=0.7
)

# ===================== FUNCTIONS =====================
def angle(a, b, c):
    """زاویه در نقطه b"""
    ba = np.array(a) - np.array(b)
    bc = np.array(c) - np.array(b)
    cosang = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosang, -1, 1)))


def resize_image(img, MAX_DIM):
    h, w = img.shape[:2]
    scale = MAX_DIM / max(h, w)   # هم بزرگ می‌کند هم کوچک
    new_w = int(w * scale)
    new_h = int(h * scale)
    img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    return img

# ===================== LOAD IMAGE =====================
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    raise FileNotFoundError(f"Cannot read image {IMAGE_PATH}")

# Resize تصویر برای فیت شدن داخل فریم
frame = resize_image(frame, MAX_DIM)
h, w = frame.shape[:2]

# ===================== YOLO DETECTION =====================
detections = []
results = yolo(frame, verbose=False)[0]
for box in results.boxes:
    if int(box.cls[0]) != 0 or box.conf[0] < CONF_THRESH:  # فقط person
        continue
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    detections.append((x1, y1, x2, y2))

# ===================== PROCESS EACH PERSON =====================
states = []

for det in detections:
    x1, y1, x2, y2 = det

    # کمی بزرگ‌تر کراپ کن
    x1_exp = max(0, x1 - 10)
    y1_exp = max(0, y1 - 10)
    x2_exp = min(w, x2 + 10)
    y2_exp = min(h, y2 + 10)

    crop = frame[y1_exp:y2_exp, x1_exp:x2_exp]
    if crop.size == 0:
        continue

    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    res = pose.process(rgb)

    state = "Unknown"
    if res.pose_landmarks:
        lm = res.pose_landmarks.landmark
        hip = lm[mp_pose.PoseLandmark.LEFT_HIP]
        knee = lm[mp_pose.PoseLandmark.LEFT_KNEE]
        ankle = lm[mp_pose.PoseLandmark.LEFT_ANKLE]
        shoulder = lm[mp_pose.PoseLandmark.LEFT_SHOULDER]

        knee_angle = angle([hip.x, hip.y], [knee.x, knee.y], [ankle.x, ankle.y])
        hip_knee_ratio = abs(hip.y - shoulder.y) / (abs(knee.y - ankle.y) + 1e-6)

        if knee_angle < KNEE_ANGLE_SIT or hip_knee_ratio < HIP_KNEE_RATIO_SIT:
            state = "Sitting"
        else:
            state = "Standing"

    states.append(state)

    # ===================== DRAW =====================
    color = (0,255,255) if state == "Sitting" else (0,255,0)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
    cv2.putText(frame, state, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 2)

# ===================== COUNT =====================
sitting_count = states.count("Sitting")
standing_count = states.count("Standing")

cv2.putText(frame, f"Sitting: {sitting_count}", (15, 35),
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2)
cv2.putText(frame, f"Standing: {standing_count}", (15, 65),
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

cv2.imshow("Sitting/Standing Detection", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

print(f"Sitting: {sitting_count}  Standing: {standing_count}")


Sitting: 2  Standing: 1
